# 02. Minimum index

**Tier:** Deep. **Why you'd use this:** A sorted schedule was rotated when the day rolled over. The smallest time is no longer at index `0`, but the two pieces are still sorted. You want the index of the minimum without scanning every slot.

**After this lesson you can:** decide which half of a rotated sorted list still holds the minimum; return that index in logarithmic steps.
About 25 minutes. **Uses:** [Lists](../../course/04-collections/01-lists-lesson.ipynb), [Loops](../../course/03-control-flow/02-loops-lesson.ipynb), [Sorted search](01-search-insert-lesson.ipynb)


## Quick review

1. In ordinary binary search, what does comparing `nums[mid]` to the target tell you?
2. What does a rotation do to a sorted list of distinct values?

<details><summary>Answers</summary>

It tells you which half can still hold the target. Rotation moves a prefix to the end, so the list is two sorted runs with one drop where the old end meets the old start. Skim [Sorted search](01-search-insert-lesson.ipynb) if the half-discard idea is fuzzy.
</details>


## Recognize it

You have a sorted list of distinct values that was rotated some unknown number of steps. You need the index of the smallest value. The prompt says "rotated sorted array", "find the minimum", or "how many times was it rotated" (that count is the same index).

Signals: almost sorted, one discontinuity, ask for the minimum's position. A look-alike that is not this pattern: fully unsorted data — then a linear scan is the honest answer.


## Why it works this way

After a rotation of a strictly increasing list, there is exactly one place where a value is followed by a smaller one (wrapping from the end back to the front counts). The minimum sits at the start of the right-hand sorted piece. Everywhere else, neighbors increase.

Compare the middle value to the right end of the current range. If `nums[mid] > nums[right]`, the drop — and therefore the minimum — lies strictly to the right of `mid`, so move `left` to `mid + 1`. If `nums[mid] <= nums[right]`, the middle sits inside a sorted suffix that includes the right end, so the minimum is at `mid` or to its left; shrink `right` to `mid`. The loop uses `left < right` and never discards `mid` on that second branch, because `mid` itself might be the minimum.

That is still binary search, but the predicate is "is the minimum to the right of mid?" rather than "is the target greater than mid?". Each step halves the search space. Time is logarithmic in the length; extra memory is constant. A linear scan also finds the minimum and needs no sortedness story, but it pays `n` comparisons every time.

Duplicates break the clean half test: when `nums[mid] == nums[right]` you cannot tell which side holds the drop. This sibling assumes distinct values so the comparison is decisive. Rotated search for a target value is a related pattern that first finds this minimum (or branches on which half is sorted) and then runs ordinary binary search inside one piece.


## When to use it

Use this when the list is a rotated sorted run of distinct values and you need the minimum's index. Prefer a linear scan when the list is unsorted or tiny. Prefer ordinary binary search when the list is sorted with no rotation. Prefer a full sort only when you must restore order for many later queries, not for a one-off minimum.


## Brute force, and why it's slow

Scan every index and keep the smallest value's position.


In [1]:
def min_index_slow(nums):
    comparisons = 0
    best = 0
    for index in range(1, len(nums)):
        comparisons += 1
        if nums[index] < nums[best]:
            best = index
    return best, comparisons


for size in (50, 200, 800):
    data = list(range(size // 2, size)) + list(range(size // 2))
    print(size, min_index_slow(data))

50 (25, 49)
200 (100, 199)
800 (400, 799)


## Key insight

If the middle value is greater than the right end, the rotation drop — and the minimum — is to the right of `mid`. Otherwise the minimum is at `mid` or to its left. Halve on that test.


## Worked example

Return the index of the minimum value in a rotated sorted list of distinct integers. A list that was never rotated returns `0`.


In [2]:
def min_index(nums: list[int]) -> int:
    # 1. Search range that still contains the minimum.
    left, right = 0, len(nums) - 1
    while left < right:
        mid = (left + right) // 2
        # 2. Middle above the right end → minimum is strictly right of mid.
        if nums[mid] > nums[right]:
            left = mid + 1
        else:
            # 3. Middle is inside a sorted suffix; keep mid as a candidate.
            right = mid
    return left


print(min_index([3, 4, 5, 1, 2]))
print(min_index([4, 5, 6, 7, 0, 1, 2]))
print(min_index([1, 2]))

3
4
0


## Trace

`[3, 4, 5, 1, 2]`:

| left | right | mid | nums[mid] | nums[right] | action |
|---|---|---|---|---|---|
| 0 | 4 | 2 | 5 | 2 | 5 > 2 → left = 3 |
| 3 | 4 | 3 | 1 | 2 | 1 ≤ 2 → right = 3 |
| 3 | 3 | — | — | — | stop → **3** |


## Cost

Each step halves the range: time grows with `log n`. Extra memory is constant. The linear scan pays about `n` comparisons; this pays a few dozen even for thousands of values.


## Variants

Not solved here: [Find Minimum in Rotated Sorted Array](../../challenges/problems/11-find-minimum-in-rotated-sorted-array.ipynb), [Search in Rotated Sorted Array](../../challenges/problems/10-search-in-rotated-sorted-array.ipynb), [Binary Search](../../challenges/problems/09-binary-search.ipynb).


## Common mistake

This bug is silent. On the branch where `nums[mid] <= nums[right]`, the code sets `right = mid - 1` and can skip past the minimum when `mid` itself was the answer. There is no traceback. The print is `4`, and the answer is `3`.


In [3]:
def min_index_wrong(nums):
    left, right = 0, len(nums) - 1
    while left < right:
        mid = (left + right) // 2
        if nums[mid] > nums[right]:
            left = mid + 1
        else:
            right = mid - 1  # drops mid, which may be the minimum
    return left


print(min_index_wrong([3, 4, 5, 1, 2]))

3


The fix sets `right = mid` so the middle stays a candidate. Run `min_index` on `[3, 4, 5, 1, 2]` and you get `3`.


In [4]:
print(min_index([3, 4, 5, 1, 2]))

3


## Python idioms

`min(range(len(nums)), key=nums.__getitem__)` finds the same index in linear time and one line. Use it when `n` is small or clarity beats the log factor. The binary version is the challenge form when the rotation story is part of the prompt.


In [5]:
def min_index_linear(nums: list[int]) -> int:
    return min(range(len(nums)), key=nums.__getitem__)


print(min_index_linear([3, 4, 5, 1, 2]))

3


## Recap

- Rotation leaves one drop; the minimum starts the right sorted piece.
- Compare middle to the right end to choose which half still holds it.
- When the middle could be the answer, set `right = mid`, not `mid - 1`.

Next: [practice](02-min-index-practice.ipynb).
